Name: Dai Dasen

SID: 1155211130

### Question 5

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Create a DataFrame for better visualization
df = pd.DataFrame(X, columns=iris.feature_names)
df['species'] = [iris.target_names[i] for i in y]

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Logistic Regression

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)

lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=5)

print(f"\nLogistic Regression Results:")
print(f"Test Accuracy: {lr_accuracy:.4f}")
print(f"Cross-Validation Accuracy: {lr_cv_scores.mean():.4f} (+/- {lr_cv_scores.std():.4f})")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=iris.target_names))

Training Logistic Regression...

Logistic Regression Results:
Test Accuracy: 0.9111
Cross-Validation Accuracy: 0.9810 (+/- 0.0233)

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.82      0.93      0.88        15
   virginica       0.92      0.80      0.86        15

    accuracy                           0.91        45
   macro avg       0.92      0.91      0.91        45
weighted avg       0.92      0.91      0.91        45



Support Vector Machine (SVM)

In [22]:
kernels = ['linear', 'rbf', 'poly']
svm_results = {}

for kernel in kernels:
    print(f"\nSVM with {kernel} kernel:")
    svm_model = SVC(kernel=kernel, random_state=42)
    svm_model.fit(X_train_scaled, y_train)
    
    y_pred_svm = svm_model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred_svm)
    cv_scores = cross_val_score(svm_model, X_train_scaled, y_train, cv=5)
    
    svm_results[kernel] = {
        'model': svm_model,
        'predictions': y_pred_svm,
        'accuracy': accuracy,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    }
    
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Cross-Validation Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Find best SVM kernel
best_kernel = max(svm_results, key=lambda k: svm_results[k]['accuracy'])
print(f"\n{'='*50}")
print(f"Best SVM kernel: {best_kernel}")
print(f"Best SVM Test Accuracy: {svm_results[best_kernel]['accuracy']:.4f}")
print(f"\nClassification Report (Best SVM):")
print(classification_report(y_test, svm_results[best_kernel]['predictions'], 
                          target_names=iris.target_names))


SVM with linear kernel:
Test Accuracy: 0.9111
Cross-Validation Accuracy: 0.9714 (+/- 0.0233)

SVM with rbf kernel:
Test Accuracy: 0.9333
Cross-Validation Accuracy: 0.9714 (+/- 0.0233)

SVM with poly kernel:
Test Accuracy: 0.8667
Cross-Validation Accuracy: 0.8952 (+/- 0.0190)

Best SVM kernel: rbf
Best SVM Test Accuracy: 0.9333

Classification Report (Best SVM):
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.88      0.93      0.90        15
   virginica       0.93      0.87      0.90        15

    accuracy                           0.93        45
   macro avg       0.93      0.93      0.93        45
weighted avg       0.93      0.93      0.93        45



Fully-Connected Neural Network

In [ ]:
# Convert data to PyTorch tensors
X_train_torch = torch.FloatTensor(X_train_scaled)
y_train_torch = torch.LongTensor(y_train)
X_test_torch = torch.FloatTensor(X_test_scaled)
y_test_torch = torch.LongTensor(y_test)

# Create DataLoader
train_dataset = TensorDataset(X_train_torch, y_train_torch)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# Define a simple fully-connected neural network
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

# Initialize model
input_size = X_train.shape[1]
hidden_size = 16
num_classes = len(np.unique(y))

simple_nn = SimpleNN(input_size, hidden_size, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(simple_nn.parameters(), lr=0.01)

# Training\
num_epochs = 200
train_losses = []
test_accuracies = []

for epoch in range(num_epochs):
    simple_nn.train()
    epoch_loss = 0
    
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = simple_nn(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    # Evaluate on test set
    simple_nn.eval()
    with torch.no_grad():
        test_outputs = simple_nn(X_test_torch)
        _, predicted = torch.max(test_outputs, 1)
        test_acc = (predicted == y_test_torch).float().mean().item()
        test_accuracies.append(test_acc)
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Test Acc: {test_acc:.4f}")

# Final evaluation
simple_nn.eval()
with torch.no_grad():
    test_outputs = simple_nn(X_test_torch)
    _, y_pred_nn = torch.max(test_outputs, 1)
    y_pred_nn = y_pred_nn.numpy()

nn_accuracy = accuracy_score(y_test, y_pred_nn)
print(f"\nSimple Neural Network Results:")
print(f"Test Accuracy: {nn_accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_nn, target_names=iris.target_names))

Training Simple Fully-Connected Neural Network...
Epoch [50/200], Loss: 0.0312, Test Acc: 0.9111
Epoch [100/200], Loss: 0.0189, Test Acc: 0.9333
Epoch [150/200], Loss: 0.0145, Test Acc: 0.9111
Epoch [200/200], Loss: 0.0114, Test Acc: 0.9111

Simple Neural Network Results:
Test Accuracy: 0.9111

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.79      1.00      0.88        15
   virginica       1.00      0.73      0.85        15

    accuracy                           0.91        45
   macro avg       0.93      0.91      0.91        45
weighted avg       0.93      0.91      0.91        45



CNN

In [26]:
# Reshape data to simulate a 1D sequence (4 features -> 4x1)
X_train_cnn = X_train_scaled.reshape(-1, 1, 4)  # (samples, channels, length)
X_test_cnn = X_test_scaled.reshape(-1, 1, 4)

X_train_cnn_torch = torch.FloatTensor(X_train_cnn)
y_train_cnn_torch = torch.LongTensor(y_train)
X_test_cnn_torch = torch.FloatTensor(X_test_cnn)
y_test_cnn_torch = torch.LongTensor(y_test)

# Create DataLoader for CNN
train_dataset_cnn = TensorDataset(X_train_cnn_torch, y_train_cnn_torch)
train_loader_cnn = DataLoader(train_dataset_cnn, batch_size=16, shuffle=True)

# Define a simple 1D CNN
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=8, kernel_size=2)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(8, 16)
        self.fc2 = nn.Linear(16, num_classes)
        
    def forward(self, x):
        out = self.conv1(x)  # Output: (batch, 8, 3)
        out = self.relu(out)
        out = self.pool(out)  # Output: (batch, 8, 1)
        out = self.flatten(out)  # Output: (batch, 8)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        return out

# Initialize CNN model
cnn_model = SimpleCNN(num_classes)
criterion_cnn = nn.CrossEntropyLoss()
optimizer_cnn = optim.Adam(cnn_model.parameters(), lr=0.01)

# Training
num_epochs_cnn = 200
train_losses_cnn = []
test_accuracies_cnn = []

for epoch in range(num_epochs_cnn):
    cnn_model.train()
    epoch_loss = 0
    
    for batch_x, batch_y in train_loader_cnn:
        optimizer_cnn.zero_grad()
        outputs = cnn_model(batch_x)
        loss = criterion_cnn(outputs, batch_y)
        loss.backward()
        optimizer_cnn.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader_cnn)
    train_losses_cnn.append(avg_loss)
    
    # Evaluate on test set
    cnn_model.eval()
    with torch.no_grad():
        test_outputs = cnn_model(X_test_cnn_torch)
        _, predicted = torch.max(test_outputs, 1)
        test_acc = (predicted == y_test_cnn_torch).float().mean().item()
        test_accuracies_cnn.append(test_acc)
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs_cnn}], Loss: {avg_loss:.4f}, Test Acc: {test_acc:.4f}")

# Final evaluation
cnn_model.eval()
with torch.no_grad():
    test_outputs = cnn_model(X_test_cnn_torch)
    _, y_pred_cnn = torch.max(test_outputs, 1)
    y_pred_cnn = y_pred_cnn.numpy()

cnn_accuracy = accuracy_score(y_test, y_pred_cnn)
print(f"\nCNN Results:")
print(f"Test Accuracy: {cnn_accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_cnn, target_names=iris.target_names))

Epoch [50/200], Loss: 0.2926, Test Acc: 0.8889
Epoch [100/200], Loss: 0.1954, Test Acc: 0.8667
Epoch [150/200], Loss: 0.1704, Test Acc: 0.8667
Epoch [200/200], Loss: 0.1410, Test Acc: 0.8889

CNN Results:
Test Accuracy: 0.8889

Classification Report:
              precision    recall  f1-score   support

      setosa       0.94      1.00      0.97        15
  versicolor       0.87      0.87      0.87        15
   virginica       0.86      0.80      0.83        15

    accuracy                           0.89        45
   macro avg       0.89      0.89      0.89        45
weighted avg       0.89      0.89      0.89        45

